# Kronos MVP：预测、微调与优化闭环

这个 notebook 先不把目标定成“完整训练一个很强的交易模型”，而是搭一个可以反复验证的 MVP：

1. 先检查本地 CSV 数据是否能被 Kronos 使用。
2. 用预训练模型跑一个 zero-shot baseline。
3. 生成最小微调配置，先做 1 epoch smoke test。
4. 再用小规模参数组合反复优化，看微调是否真的改善预测指标。

核心原则：先证明流程能闭环，再扩大数据和训练规模。

## 1. 微调到底调的是什么

Kronos 里有两个容易混在一起的对象：

**Tokenizer**：把连续的 K 线数值，例如 `open/high/low/close/volume/amount`，压缩成离散 token。微调 tokenizer，本质是在调整“如何把本地市场数据切成模型能理解的离散符号”。这更像适配数据分布，风险是如果数据不够，tokenizer 可能把原来已经学好的通用表示弄坏。

**Base model / predictor**：拿历史 token 和时间特征，预测未来 token。微调 predictor，本质是在调整“看到过去一段行情后，如何生成未来行情 token”。这更贴近我们当前的目标，也更适合作为 MVP 的第一步。

所以这里的 MVP 策略是：**先固定 tokenizer，优先微调 predictor；如果 predictor 微调确实有效，再考虑 tokenizer。**

## 2. 实验路线

我们先按下面的顺序推进：

1. 数据体检：列名、时间戳、缺失值、样本长度。
2. Zero-shot：直接用 `NeoQuasar/Kronos-Tokenizer-base` + `NeoQuasar/Kronos-small` 预测。
3. Smoke fine-tune：用极小训练轮数确认训练脚本能跑通。
4. Basemodel fine-tune：主要调 predictor 的学习率、epoch、batch size。
5. 评估：至少看 MAE、RMSE、MAPE、方向准确率；后面再接简单回测。

注意：金融预测里，单次预测好看不代表策略有效，所以最终要用时间切分的验证集和交易成本约束来判断。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Kronos_projest":
    PROJECT_ROOT = Path("/home/hmx42/Projects/Kronos_projest")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "my test/main/mvp_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, OUTPUT_DIR

## 3. 选择一份 MVP 数据

优先用仓库里微调示例数据。如果你之后想换成自己的 A 股或港股数据，只需要把 `data_path` 改成对应 CSV。

In [ ]:
candidate_paths = [
    PROJECT_ROOT / "finetune_csv/data/HK_ali_09988_kline_5min_all.csv",
    PROJECT_ROOT / "data/XSHG_5min_600977.csv",
    PROJECT_ROOT / "tests/data/regression_input.csv",
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("没有找到可用 CSV，请手动设置 data_path。")

df = pd.read_csv(data_path)
print("using:", data_path)
print("shape:", df.shape)
df.head()

In [ ]:
required = ["timestamps", "open", "high", "low", "close"]
optional = ["volume", "amount"]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少必要列: {missing}")

df = df.copy()
df["timestamps"] = pd.to_datetime(df["timestamps"])
for col in optional:
    if col not in df.columns:
        df[col] = 0.0

feature_cols = ["open", "high", "low", "close", "volume", "amount"]
df = df.sort_values("timestamps").reset_index(drop=True)

summary = {
    "rows": len(df),
    "start": str(df["timestamps"].min()),
    "end": str(df["timestamps"].max()),
    "missing_values": int(df[["timestamps"] + feature_cols].isna().sum().sum()),
    "duplicated_timestamps": int(df["timestamps"].duplicated().sum()),
}

summary

## 4. 评估函数

这里先不做复杂回测，只看预测本身：

- `MAE`：平均绝对误差。
- `RMSE`：对大误差更敏感。
- `MAPE`：相对误差。
- `direction_accuracy`：预测涨跌方向是否正确。交易里这个指标通常比绝对价格误差更接近策略价值。

In [ ]:
def regression_metrics(pred_close, true_close):
    pred_close = np.asarray(pred_close, dtype=float)
    true_close = np.asarray(true_close, dtype=float)
    err = pred_close - true_close
    return {
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "mape": float(np.mean(np.abs(err) / np.maximum(np.abs(true_close), 1e-8))),
    }


def direction_accuracy(pred_close, true_close, last_context_close=None):
    pred_close = np.asarray(pred_close, dtype=float)
    true_close = np.asarray(true_close, dtype=float)
    if last_context_close is None:
        pred_ret = np.diff(pred_close)
        true_ret = np.diff(true_close)
    else:
        pred_ret = np.diff(np.r_[last_context_close, pred_close])
        true_ret = np.diff(np.r_[last_context_close, true_close])
    return float(np.mean(np.sign(pred_ret) == np.sign(true_ret)))


def evaluate_forecast(forecast_df, true_df, last_context_close):
    metrics = regression_metrics(forecast_df["close"], true_df["close"])
    metrics["direction_accuracy"] = direction_accuracy(
        forecast_df["close"],
        true_df["close"],
        last_context_close=last_context_close,
    )
    return metrics

## 5. Zero-shot baseline

这一段直接加载预训练 tokenizer 和 predictor，不做任何微调。它的作用是给后面的微调一个参照物：如果微调后没有超过它，就说明微调设置、数据切分或目标设计还需要调整。

In [ ]:
RUN_ZERO_SHOT = False

lookback = min(400, len(df) // 2)
pred_len = min(120, len(df) - lookback)

if pred_len <= 0:
    raise ValueError("数据太短，无法切出预测区间。")

x_df = df.loc[: lookback - 1, feature_cols]
x_timestamp = df.loc[: lookback - 1, "timestamps"]
y_timestamp = df.loc[lookback : lookback + pred_len - 1, "timestamps"]
true_df = df.loc[lookback : lookback + pred_len - 1, feature_cols].reset_index(drop=True)
last_context_close = float(df.loc[lookback - 1, "close"])

print({"lookback": lookback, "pred_len": pred_len})

if RUN_ZERO_SHOT:
    from model import Kronos, KronosTokenizer, KronosPredictor

    tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
    model = Kronos.from_pretrained("NeoQuasar/Kronos-small")
    predictor = KronosPredictor(model, tokenizer, max_context=512)

    forecast = predictor.predict(x_df, x_timestamp, y_timestamp, pred_len=pred_len)
    forecast_path = OUTPUT_DIR / "zero_shot_forecast.csv"
    forecast.to_csv(forecast_path, index=False)

    metrics = evaluate_forecast(forecast, true_df, last_context_close)
    metrics_path = OUTPUT_DIR / "zero_shot_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    print(metrics)
    print("saved:", forecast_path, metrics_path)
else:
    print("RUN_ZERO_SHOT=False，当前只展示代码逻辑，不加载模型。")

## 6. 生成 MVP 微调配置

这里先生成一个 1 epoch 的 smoke-test 配置。它不是为了追求效果，而是为了确认：

1. 数据能被训练脚本读取。
2. 模型能正常 forward/backward。
3. checkpoint 能保存到本地。

当前仓库的 `train_sequential.py --skip-tokenizer` 对 tokenizer 路径有本地存在性检查，所以最稳妥的第一版是 tokenizer 和 predictor 都跑 1 epoch。等流程通了，再改成只微调 predictor。

In [ ]:
try:
    import torch
    use_cuda = bool(torch.cuda.is_available())
except Exception:
    use_cuda = False

config = {
    "data": {
        "data_path": str(data_path),
        "lookback_window": 256,
        "predict_window": 32,
        "max_context": 512,
        "clip": 5.0,
        "train_ratio": 0.8,
        "val_ratio": 0.1,
        "test_ratio": 0.1,
    },
    "training": {
        "tokenizer_epochs": 1,
        "basemodel_epochs": 1,
        "batch_size": 4,
        "log_interval": 10,
        "num_workers": 0,
        "seed": 42,
        "tokenizer_learning_rate": 2e-4,
        "predictor_learning_rate": 1e-6,
        "adam_beta1": 0.9,
        "adam_beta2": 0.95,
        "adam_weight_decay": 0.1,
        "accumulation_steps": 1,
    },
    "model_paths": {
        "pretrained_tokenizer": "NeoQuasar/Kronos-Tokenizer-base",
        "pretrained_predictor": "NeoQuasar/Kronos-small",
        "exp_name": "mvp_smoke_tokenizer_predictor_1epoch",
        "base_path": str(OUTPUT_DIR / "finetuned"),
        "base_save_path": "",
        "finetuned_tokenizer": "",
        "tokenizer_save_name": "tokenizer",
        "basemodel_save_name": "basemodel",
    },
    "experiment": {
        "name": "kronos_mvp_finetune",
        "description": "MVP smoke test on custom CSV data",
        "use_comet": False,
        "train_tokenizer": True,
        "train_basemodel": True,
        "skip_existing": False,
        "pre_trained_tokenizer": True,
        "pre_trained_predictor": True,
    },
    "device": {
        "use_cuda": use_cuda,
        "device_id": 0,
    },
}

import yaml

config_path = OUTPUT_DIR / "config_mvp_smoke.yaml"
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print("saved:", config_path)
print(config_path.read_text(encoding="utf-8")[:1200])

## 7. 运行 smoke test

下面的 cell 默认不会直接跑训练。你准备运行时，把 `RUN_TRAINING` 改成 `True`。

如果机器显存紧张，优先把配置里的 `batch_size` 从 4 降到 2；如果训练太慢，先把 `lookback_window` 从 256 降到 128。

In [ ]:
RUN_TRAINING = False

cmd = [
    sys.executable,
    "train_sequential.py",
    "--config",
    str(config_path),
]

print("working dir:", PROJECT_ROOT / "finetune_csv")
print("command:", " ".join(cmd))

if RUN_TRAINING:
    result = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT / "finetune_csv",
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"training failed with code {result.returncode}")
else:
    print("RUN_TRAINING=False，当前不启动训练。")

## 8. 只微调 predictor 的方案

理想的 MVP 是：tokenizer 固定，主要微调 predictor。但当前脚本在 `--skip-tokenizer` 时要求 `finetuned_tokenizer` 是一个本地存在的路径。

两种推进方式：

1. **最少改代码**：先跑一次 tokenizer 1 epoch，得到本地 tokenizer checkpoint，再用它微调 predictor。
2. **更干净的代码改造**：把 `train_sequential.py` 里的本地路径检查改成允许 Hugging Face ID，例如 `NeoQuasar/Kronos-Tokenizer-base`。这样就能真正固定原始 tokenizer，只训练 predictor。

在正式报告里，建议诚实写成：MVP 阶段优先探索 predictor 微调；tokenizer 微调作为后续扩展。

In [ ]:
predictor_only_config = json.loads(json.dumps(config))
predictor_only_config["model_paths"]["exp_name"] = "mvp_predictor_only_1epoch"
predictor_only_config["experiment"]["train_tokenizer"] = False
predictor_only_config["experiment"]["train_basemodel"] = True

# 这里先指向 smoke test 产生的本地 tokenizer。只有当上面的 smoke test 跑完后，这个路径才会存在。
predictor_only_config["model_paths"]["finetuned_tokenizer"] = str(
    OUTPUT_DIR / "finetuned/mvp_smoke_tokenizer_predictor_1epoch/tokenizer/best_model"
)

predictor_only_config_path = OUTPUT_DIR / "config_mvp_predictor_only.yaml"
predictor_only_config_path.write_text(
    yaml.safe_dump(predictor_only_config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("saved:", predictor_only_config_path)
print("tokenizer path exists:", Path(predictor_only_config["model_paths"]["finetuned_tokenizer"]).exists())
print("command:")
print(
    "cd", PROJECT_ROOT / "finetune_csv", "&&",
    sys.executable, "train_sequential.py --config", predictor_only_config_path, "--skip-tokenizer"
)

## 9. 多次尝试优化：小网格搜索

优化不要一开始铺太大。第一轮只动三个最关键的变量：

- `predictor_learning_rate`：学习率太大容易破坏预训练能力，太小则几乎不动。
- `basemodel_epochs`：先看 1、3、5 epoch 的趋势。
- `lookback_window`：不同市场频率下，有效历史长度可能不同。

下面只生成配置和命令，不自动执行。等 smoke test 过了，再挑 2-4 组跑。

In [ ]:
grid = []
for lr in [5e-7, 1e-6, 3e-6]:
    for epochs in [1, 3]:
        for lookback_window in [256, 512]:
            grid.append({
                "predictor_learning_rate": lr,
                "basemodel_epochs": epochs,
                "lookback_window": lookback_window,
            })

grid_dir = OUTPUT_DIR / "grid_configs"
grid_dir.mkdir(parents=True, exist_ok=True)

commands = []
for i, params in enumerate(grid, start=1):
    cfg = json.loads(json.dumps(predictor_only_config))
    cfg["model_paths"]["exp_name"] = (
        f"grid_{i:02d}_lr{params['predictor_learning_rate']}_"
        f"ep{params['basemodel_epochs']}_lb{params['lookback_window']}"
    )
    cfg["training"]["predictor_learning_rate"] = params["predictor_learning_rate"]
    cfg["training"]["basemodel_epochs"] = params["basemodel_epochs"]
    cfg["data"]["lookback_window"] = params["lookback_window"]

    path = grid_dir / f"{cfg['model_paths']['exp_name']}.yaml"
    path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    commands.append(f"cd {PROJECT_ROOT / 'finetune_csv'} && {sys.executable} train_sequential.py --config '{path}' --skip-tokenizer")

commands_path = OUTPUT_DIR / "grid_commands.sh"
commands_path.write_text("\n".join(commands) + "\n", encoding="utf-8")

print(f"generated {len(commands)} configs")
print("commands saved to:", commands_path)
print("first 3 commands:")
print("\n".join(commands[:3]))

## 10. 微调后重新评估

训练完成后，应该用同一段验证/测试窗口比较：

- zero-shot 模型
- smoke fine-tuned 模型
- 不同参数组合的 predictor fine-tuned 模型

这里只放评估入口骨架。具体 checkpoint 路径要根据训练输出选择，例如：

`my test/main/mvp_outputs/finetuned/<exp_name>/basemodel/best_model`

In [ ]:
RUN_FINETUNED_EVAL = False

finetuned_model_path = OUTPUT_DIR / "finetuned/mvp_predictor_only_1epoch/basemodel/best_model"
finetuned_tokenizer_path = OUTPUT_DIR / "finetuned/mvp_smoke_tokenizer_predictor_1epoch/tokenizer/best_model"

if RUN_FINETUNED_EVAL:
    from model import Kronos, KronosTokenizer, KronosPredictor

    tokenizer = KronosTokenizer.from_pretrained(str(finetuned_tokenizer_path))
    model = Kronos.from_pretrained(str(finetuned_model_path))
    predictor = KronosPredictor(model, tokenizer, max_context=512)

    forecast = predictor.predict(x_df, x_timestamp, y_timestamp, pred_len=pred_len)
    metrics = evaluate_forecast(forecast, true_df, last_context_close)
    print(metrics)
else:
    print("RUN_FINETUNED_EVAL=False，训练完后再打开这一段。")

## 11. 当前结论模板

如果要写进 README 或报告，可以这样表达：

> 本项目当前的 MVP 目标不是声称已经完成 Kronos 的充分微调，而是建立了从本地 K 线数据、预训练预测、微调配置生成、checkpoint 保存到指标评估的闭环。第一阶段优先固定 tokenizer，微调 predictor，以验证预训练时序模型能否适配本地 A/HK 市场数据。后续会在多股票、多时间窗口和带交易成本的回测环境下评估微调收益。

判断是否继续扩大的标准：

1. 微调后 MAE/RMSE 至少不显著变差。
2. 方向准确率相对 zero-shot 有稳定提升。
3. 简单交易规则下，扣除成本后仍有改善。
4. 多个时间段复现，而不是只在单一窗口好看。